In [ ]:
import h5py
import numpy as np

import scipy
import scipy.constants as constants

from IPython.display import clear_output

from helper_functions import (sphere_idx, cylinder_idx, write_text)

import spimage

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn as sns
sns.set_theme()
 
import sys, time
import os, os.path

pi = scipy.constants.pi
e = constants.elementary_charge
h = constants.Planck
c = constants.speed_of_light

<h2> Loading 3D model to phase </h2> 

In [ ]:
subBg = False

diagPlotEMC = True
diagPlotSIM = False

emc_file = 'emc/protein_with_water_debug_0001/data_50k/prot_only_3/output_380.h5'
with h5py.File(emc_file, "r") as f_ptr:
    I_emc = np.squeeze(f_ptr['intens'][:])
    W_emc = np.squeeze(f_ptr['inter_weight'][:])
    scale_emc = f_ptr['scale'][:]
    llk_emc = f_ptr['likelihood'][:]

I_emc = I_emc[:-1,:-1,:-1]
W_emc = W_emc[:-1,:-1,:-1]

if subBg:
    emc_bg_file = 'emc/water_only_debug_0001/data_50k/2/output_400.h5'
    with h5py.File(emc_bg_file,'r') as f_bg:
        I_emc_bg = np.squeeze(f_bg['intens'][:])
        scale_bg = f_bg['scale'][:]

        llk_bg = f_bg['likelihood'][:]

    I_emc_bg = I_emc_bg[:-1,:-1,:-1]

    frame = I_emc - I_emc_bg
else:
    frame = I_emc

center = frame.shape[0]//2

sphere_rad = 65

mask_emc = sphere_idx(shape=frame.shape,radius=sphere_rad, position=(center, center, center))
mask_emc = np.logical_and(mask_emc, W_emc.astype(np.bool_))
frac_emc_good = mask_emc.sum() / (mask_emc.shape[0]**3)

mask_pos = frame >= 0.0
mask_all = mask_emc & mask_pos

file = emc_file
npats = file.split(sep='/')[2].split(sep='_')[1]
sType = file.split(sep="/")[2]
pType = 'emc_'+file.split(sep='/')[3]
print(f'Phasing {pType} model ({npats} patterns)!')
    
if diagPlotEMC:
    frame_scaling = 1.0
    test_slice = center
    
    xy_obj = frame[:,:,test_slice] ** frame_scaling
    xz_obj = frame[:,test_slice,:] ** frame_scaling
    yz_obj = frame[test_slice,:,:] ** frame_scaling

    fig_handle = plt.figure(constrained_layout = True, dpi = 200)
    fig_handle.patch.set_facecolor('gray')
    spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
    cm = 'viridis'
    max_v = 0.3

    ax_0 = fig_handle.add_subplot(spec_handle[0,0])
    im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_0.set_xticks([])
    ax_0.set_yticks([])
    minv, maxv = im_0.get_clim()
    ax_0.set_title(f'XY-{test_slice}',weight='bold',fontsize=6) 
    c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.01, shrink=0.43) 
    c_bar_0.set_ticks([minv,maxv*0.5,maxv]) 

    ax_1 = fig_handle.add_subplot(spec_handle[0,1]) 
    im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_1.set_xticks([]) 
    ax_1.set_yticks([]) 
    minv, maxv = im_1.get_clim()
    ax_1.set_title(f'XZ-{test_slice}',weight='bold',fontsize=6) 

    ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
    im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
    ax_2.set_xticks([])
    ax_2.set_yticks([])
    minv, maxv = im_2.get_clim()
    ax_2.set_title(f' YZ-{test_slice}',weight='bold',fontsize=6);
    
    fig_handle = plt.figure(constrained_layout = True, dpi = 200)
    fig_handle.patch.set_facecolor('gray')
    spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
    cm = 'gray'
    max_v = 1.0

    xy_obj = mask_emc[:,:,test_slice]
    xz_obj = mask_emc[:,test_slice,:]
    yz_obj = mask_emc[test_slice,:,:]
    
    ax_0 = fig_handle.add_subplot(spec_handle[0,0])
    im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
    ax_0.set_xticks([])
    ax_0.set_yticks([]) 
    minv, maxv = im_0.get_clim()
    ax_0.set_title(f'(EMC) XY-{test_slice}',weight='bold',fontsize=6) 
    c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.01, shrink=0.43) 
    c_bar_0.set_ticks([minv,maxv]) 

    ax_1 = fig_handle.add_subplot(spec_handle[0,1]) 
    im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_1.set_xticks([]) 
    ax_1.set_yticks([]) 
    minv, maxv = im_1.get_clim() 
    ax_1.set_title(f'(EMC) XZ-{test_slice}',weight='bold',fontsize=6) 

    ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
    im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
    ax_2.set_xticks([]) 
    ax_2.set_yticks([])
    minv, maxv = im_2.get_clim()
    ax_2.set_title(f'(EMC) YZ-{test_slice}',weight='bold',fontsize=6);
    
    fig_handle = plt.figure(constrained_layout = True, dpi = 200)
    fig_handle.patch.set_facecolor('gray')
    spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)

    xy_obj = mask_pos[:,:,test_slice]
    xz_obj = mask_pos[:,test_slice,:]
    yz_obj = mask_pos[test_slice,:,:]
    
    ax_0 = fig_handle.add_subplot(spec_handle[0,0])
    im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
    ax_0.set_xticks([])
    ax_0.set_yticks([]) 
    minv, maxv = im_0.get_clim()
    ax_0.set_title(f'(Pos) XY-{test_slice}',weight='bold',fontsize=6) 
    c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.01, shrink=0.43) 
    c_bar_0.set_ticks([minv,maxv]) 

    ax_1 = fig_handle.add_subplot(spec_handle[0,1]) 
    im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_1.set_xticks([]) 
    ax_1.set_yticks([]) 
    minv, maxv = im_1.get_clim() 
    ax_1.set_title(f'(Pos) XZ-{test_slice}',weight='bold',fontsize=6) 

    ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
    im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_2.set_xticks([]) 
    ax_2.set_yticks([])
    inv, maxv = im_2.get_clim()
    ax_2.set_title(f'(Pos) YZ-{test_slice}',weight='bold',fontsize=6);
    
    fig_handle = plt.figure(constrained_layout = True, dpi = 200)
    fig_handle.patch.set_facecolor('gray')
    spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
    
    xy_obj = mask_all[:,:,test_slice]
    xz_obj = mask_all[:,test_slice,:]
    yz_obj = mask_all[test_slice,:,:]

    ax_0 = fig_handle.add_subplot(spec_handle[0,0])
    im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
    ax_0.set_xticks([])
    ax_0.set_yticks([]) 
    inv, maxv = im_0.get_clim()
    ax_0.set_title(f'(EMC+Pos) XY-{test_slice}',weight='bold',fontsize=6) 
    c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.01, shrink=0.43) 
    c_bar_0.set_ticks([minv,maxv]) 

    ax_1 = fig_handle.add_subplot(spec_handle[0,1]) 
    im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_1.set_xticks([]) 
    ax_1.set_yticks([])
    minv, maxv = im_1.get_clim() 
    ax_1.set_title(f'(EMC+Pos) XZ-{test_slice}',weight='bold',fontsize=6) 

    ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
    im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_2.set_xticks([])
    ax_2.set_yticks([])
    minv, maxv = im_2.get_clim()
    ax_2.set_title(f'(EMC+Pos) YZ-{test_slice}',weight='bold',fontsize=6);
    
num_pix = np.prod(frame.shape)
write_text(f'Mean intensity: {frame.mean()}\n')
write_text(f'Min intensity: {frame.min()}\n')
write_text(f'Max intensity: {frame.max()}\n')
write_text(f'Number of negative voxels: {num_pix-mask_pos.sum()}\n')
write_text(f'Fraction of positive voxels: {mask_pos.sum()}/{num_pix}\n')
write_text(f'Percentage of positive voxels: {(mask_pos.sum()/num_pix)*100}%\n')
    
e_photon_eV = 9000
lambda_photon = (h * c) / (e_photon_eV * e)
d_detector = 1.0
s_pixel = 2400e-6

In [ ]:
dimX = frame.shape[0]
dimY = frame.shape[1]
dimZ = frame.shape[2]

pixel_num = dimX - dimX//2
theta_pixel = 0.5 * np.arctan((pixel_num*s_pixel)/d_detector)
resolution = lambda_photon/(2.0 * np.sin(theta_pixel))
voxel_size = 0.5 * resolution # size of voxel in m
write_text(f'The resolution (real-space): {resolution*1e10} Å\n')
write_text(f'The voxel size (real-space): {voxel_size*1e10} Å')

# Cylindrical support and volume fraction estimate - from "GroEL-Mediated Protein Folding: Making the Impossible, Possible"
print('Cylindrical support estimate!')
H_part = 14.7e-9
D_part = 13.7e-9
R_part = D_part/2

H_particle_vox = H_part/voxel_size
R_particle_vox = R_part/voxel_size

volume_fraction_cyl = (H_particle_vox*pi*(R_particle_vox**2))/(dimX*dimY*dimZ)
print(f'Support volume: {H_part*pi*(R_part**2)*1e27} nm^3')
print(f'Support volume: {H_particle_vox*pi*(R_particle_vox**2)} vox^3')
print(f'The final volume fraction is: {volume_fraction_cyl}')
print(f'The final volume percentage is: {volume_fraction_cyl*100}%')
print(f'The height is: {H_particle_vox} voxels')
print(f'The diameter is: {2*R_particle_vox} voxels')

support_cyl = cylinder_idx(shape=frame.shape,height=H_particle_vox,radius=R_particle_vox,position=(dimX//2,dimY//2,dimZ//2))
vox_cyl = support_cyl.sum()
print(f'The number of object voxels: {vox_cyl}')
print(f'The number of non-object voxels: {dimX*dimY*dimZ-vox_cyl}\n')
vol_frac = volume_fraction_cyl

<h2> Performing phase retrieval</h2> 
Three commonly-used algorithms in iterative phasing are er, hio, raar, and difference map (diffmap). The former is used more nowadays near the end of the phasing loop, due to its ability to 
aggresively reduce the error. Common in all, the object support is used for the object size/extent. Shrinkwrap is used by most algorithms.
An overview of the commonly-used algorithms follows below (prime indicates current object guess). 
The equations will be given in terms of the k-th iteration:
<h4> er </h4>

$$o_{k+1}=\begin{cases} 
P_M\,o_k & \text{if in support} \\
0 & \text {if not in support}
\end{cases}$$

<h4> hio </h4>
$$o_{k+1}=\begin{cases} 
P_M\,o_k & \text{if in support} \\
o_{k} - \beta\,P_M\,o_k & \text {if not in support}
\end{cases}$$
For hio, the interpretation of the feedback parameter $\beta$ is to place more emphasis on the new estimate or the current estimate (at iteration k). 
The analogy can be seen from that a large $\beta$ will yield large oscillations and a slow convergence, whereas a much smaller value will yield slower
convergence to the solution (not necessarily global solution). If $\beta$ is selected to be below 1, convergence is smooth towards the solution. Note that
this $\beta$ is not equivalent in terms of interpretation to the $\beta$ in raar. Commonly values near 0.9 are used. The $P_M$ is the Fourier amplitude constraint
used (our intensity measurements).

<h4> raar </h4>
We begin by defining the reflectors ($R$) on the support set S (convex set) and Fourier domain constraint $M$ (nonconvex set), where $P$ is the projection operator. 
The projection operator $P$ projects a signal onto the set $M$ (Fourier amplitude constraint in fancy notation basically).

$$ R_S=2\times P_S-I $$
$$ R_M=2\times P_M-I $$
where $I$ is the identity operator.
Finally, this yields the following algorithm:

$$o_{k+1}=\begin{cases}
P_M\,o_k & \text{if in support} \\
\beta\,o_k+(1-2\,\beta)\,P_M\,o_k\, & \text {if not in support}
\end{cases}$$
Some values for $\beta$ are suggested in the original RAAR paper. For a static feedback parameter, the best solution is achieved with a value of 0.87. Too low values of the parameter 
lead to a worse reconstruction. For values close to 1 the solution is eventually better, but the convergence proceeds much slower. The
variable feedback parameter used was suggested to be $0.75$ to $1.0$. It was also found that to stabilise a solution the feedback parameter should not be set to $1.0$ 
exactly, but instead to a lower value - say $0.99999$. 

<h4> diffmap </h4>
A more general algorithm where hio is a special case of difference map. For certain choices $\beta$ raar is a special case of diffmap as well. However,
there are some cases where raar is not equivalent to diffmap (when $\beta\neq 1$). 

In [ ]:
alg = 'raar'

n_recons = 3
niter_alg = 500
niter_er = 450
niter_store = 2

beta_start = 0.80
beta_end = 0.85

i_frac, f_frac = 1.1, 1.01
volume_i, volume_f = i_frac * vol_frac, f_frac * vol_frac

blur_i, blur_f = 1.5, 1.0
supp_update = 20

niter_store_errors = 10

recon_intens = frame.copy()
recon_mask = mask_emc.copy()

# Initialisation of arrays
def_rng = np.random.default_rng()

recon_real_array = np.zeros(shape=(n_recons,niter_store,*recon_intens.shape)) 
recon_phase_array = np.zeros(shape=(n_recons,niter_store,*recon_intens.shape))

recon_real_array_nosupp = np.zeros(shape=(n_recons,niter_store,*recon_intens.shape)) 
recon_phase_array_nosupp = np.zeros(shape=(n_recons,niter_store,*recon_intens.shape)) 

recon_fourier_array = np.zeros(shape=(n_recons,niter_store,*recon_intens.shape)) 
recon_fourier_poiss_array = np.zeros(shape=(n_recons,niter_store,*recon_intens.shape)) 

support_array = np.zeros(shape=(n_recons,niter_store,*recon_intens.shape))

error_real_array = np.zeros(shape=(n_recons,niter_store_errors))
error_fourier_array = np.zeros(shape=(n_recons,niter_store_errors))

constraints_list = ['enforce_positivity', 'enforce_real']

fourier_mask = 'mask_emc'
vnum = '1'

# Phasing loop
time_now = time.localtime(time.time())
for i in range (n_recons):
    write_text(f'\rRunning reconstruction {i+1}/{n_recons}...')
    phaser = spimage.Reconstructor()
    
    # Initialising iterations and output parameters 
    phaser.set_number_of_iterations(niter_alg+niter_er)
    phaser.set_number_of_outputs_images(niter_store)  
    phaser.set_number_of_outputs_scores(niter_store_errors)
    
    # Initial support
    phaser.set_initial_support(support_mask=support_cyl)
    
    # Fourier space mask - only for EMC reconstructions
    phaser.set_mask(recon_mask)
    
    # Setting intensity to be phased
    phaser.set_intensities(recon_intens)
    
    # Support algorithm 
    phaser.append_support_algorithm('area',blur_init=blur_i,blur_final=blur_f,area_init=volume_i,area_final=volume_f,
                                    update_period=supp_update,number_of_iterations=niter_alg+niter_er)
    
    # Phasing algorithms
    if alg == 'diffmap':
        phaser.append_phasing_algorithm(alg, constraints=constraints_list, beta_init=beta_start, beta_final=beta_end, 
                                        number_of_iterations=niter_alg, gamma1=-1/beta_start,gamma2=(3-beta_start)/(2*beta_start))
    else:
        phaser.append_phasing_algorithm(alg, constraints=constraints_list, beta_init=beta_start, beta_final=beta_end,
                                        number_of_iterations=niter_alg)
    
    phaser.append_phasing_algorithm('er', constraints=constraints_list, number_of_iterations=niter_er)

    output = phaser.reconstruct()

    # Retrieving phasing results
    recon_real = output['real_space']
    recon_fourier = output['fourier_space']
    support = output['support']
    error_real = output['real_error']
    error_fourier = output['fourier_error']
    
    # Enforcing support constraint for density
    object_density = np.abs(recon_real)
    object_density[support==False] = 0.
    
    object_phase = np.angle(recon_real)
    object_phase[support==False] = 0.
    
    # Not enforcing support constraint for density
    object_density_nsp = np.abs(recon_real)
    object_phase_nsp = np.angle(recon_real)
    
    object_fourier = np.abs(recon_fourier)**2
    object_fourier_poiss = def_rng.poisson(lam=object_fourier)
    
    # Storing reconstructions results in arrays
    recon_real_array[i] = object_density
    recon_phase_array[i] = object_phase
    recon_real_array_nosupp[i] = object_density_nsp
    recon_phase_array_nosupp[i] = object_phase_nsp
    
    recon_fourier_array[i] = object_fourier
    recon_fourier_poiss_array[i] = object_fourier_poiss
    support_array[i] = support
    error_real_array[i] = error_real
    error_fourier_array[i] = error_fourier
     
    clear_output(wait=True)
                
write_text(f'\rAll {n_recons} reconstruction(s) finished!!!')

<h2> Inspecting slices of 3D phase reconstruction </h2> 
Here we show the retrieved densities with and without application of the support constraint. 
The density should be low enough that thresholding on the values there should result in a perfect fitting mask
resembling the shape of GroEL. 

In [ ]:
SupportApplied = False
MaximumProjection = True

center = dimX//2
zoom = 35
im_slice = dimX//2

rec_num = 0

dens_scaling = 1.0
yz_obj = recon_real_array_nosupp[rec_num][-1][center-zoom:center+zoom,center-zoom:center+zoom,im_slice] ** dens_scaling
xz_obj = recon_real_array_nosupp[rec_num][-1][center-zoom:center+zoom,im_slice,center-zoom:center+zoom] ** dens_scaling
xy_obj = recon_real_array_nosupp[rec_num][-1][im_slice,center-zoom:center+zoom,center-zoom:center+zoom] ** dens_scaling

fig_handle = plt.figure(constrained_layout = True, dpi = 280)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 3, ncols = 3)
plt.suptitle(f'Slices of object amplitude and phase: reconstruction {rec_num+1}/{n_recons}')
cm = 'viridis'
max_v = 0.0006

# Reconstructed electron densities
ax_0 = fig_handle.add_subplot(spec_handle[0,0])
im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
ax_0.set_xticks([])
ax_0.set_yticks([])
minv, maxv = im_0.get_clim()
ax_0.set_title(f'(XY-{im_slice}/{dimX})',weight='bold',fontsize=6) 
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05) 
c_bar_0.set_label('electron density in a.u.', weight='bold',fontsize=6)
c_bar_0.set_ticks([minv,maxv*0.5,maxv]) 

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_1.set_xticks([]) 
ax_1.set_yticks([]) 
minv, maxv = im_1.get_clim() 
ax_1.set_title(f'(XZ-{im_slice}/{dimY})',weight='bold',fontsize=6)

ax_2 = fig_handle.add_subplot(spec_handle[0,2])
im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_2.set_xticks([]) 
ax_2.set_yticks([]) 
minv, maxv = im_2.get_clim() 
ax_2.set_title(f'(YZ-{im_slice}/{dimZ})',weight='bold',fontsize=6)

if MaximumProjection:
    # Maximum projection electron densities
    xy_obj = recon_real_array_nosupp[rec_num][-1].max(axis=0)[center-zoom:center+zoom,center-zoom:center+zoom]
    xz_obj = recon_real_array_nosupp[rec_num][-1].max(axis=1)[center-zoom:center+zoom,center-zoom:center+zoom]
    yz_obj = recon_real_array_nosupp[rec_num][-1].max(axis=2)[center-zoom:center+zoom,center-zoom:center+zoom]

    ax_6 = fig_handle.add_subplot(spec_handle[2,0])
    im_6 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v*3,cmap=cm,interpolation=None)
    ax_6.set_xticks([])
    ax_6.set_yticks([])
    minv, maxv = im_6.get_clim()
    ax_6.set_title('XY',weight='bold',fontsize=6) 
    c_bar_6 = plt.colorbar(im_6, ax=ax_6,fraction=0.05)
    c_bar_6.set_label('electron density in a.u.', weight='bold',fontsize=6)
    c_bar_6.set_ticks([minv,maxv*0.5,maxv])

    ax_7 = fig_handle.add_subplot(spec_handle[2,1])
    im_7 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v*3,cmap=cm,interpolation=None) 
    ax_7.set_xticks([]) 
    ax_7.set_yticks([]) 
    minv, maxv = im_7.get_clim() 
    ax_7.set_title('XZ',weight='bold',fontsize=6)

    ax_8 = fig_handle.add_subplot(spec_handle[2,2])
    im_8 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v*3,cmap=cm,interpolation=None) 
    ax_8.set_xticks([])
    ax_8.set_yticks([]) 
    minv, maxv = im_8.get_clim() 
    ax_8.set_title('YZ',weight='bold',fontsize=6)

# Reconstructed electron density phases 
yz_diff = recon_phase_array_nosupp[rec_num][-1][center-zoom:center+zoom,center-zoom:center+zoom,im_slice]
xz_diff = recon_phase_array_nosupp[rec_num][-1][center-zoom:center+zoom,im_slice,center-zoom:center+zoom]
xy_diff = recon_phase_array_nosupp[rec_num][-1][im_slice,center-zoom:center+zoom,center-zoom:center+zoom]

ax_3 = fig_handle.add_subplot(spec_handle[1,0]) 
im_3 = plt.imshow(xy_diff,vmin=-pi,vmax=pi,cmap='twilight_shifted',interpolation=None)
ax_3.set_xticks([])
ax_3.set_yticks([])
minv, maxv = im_3.get_clim()
ax_3.set_title(f'(XY-{im_slice}/{dimX})',weight='bold',fontsize=6)
c_bar_3 = plt.colorbar(im_3, ax=ax_3,fraction=0.02) 
c_bar_3.set_label('phase in radians', weight='bold',fontsize=6)
c_bar_3.set_ticks([-3,0,3])

ax_4 = fig_handle.add_subplot(spec_handle[1,1]) 
im_4 = plt.imshow(xz_diff,vmin=-pi,vmax=pi,cmap='twilight_shifted',interpolation=None) 
ax_4.set_xticks([]) 
ax_4.set_yticks([])

minv, maxv = im_4.get_clim() 
ax_4.set_title(f'(XZ-{im_slice}/{dimY})',weight='bold',fontsize=6) 

ax_5 = fig_handle.add_subplot(spec_handle[1,2]) 
im_5 = plt.imshow(yz_diff,vmin=-pi,vmax=pi,cmap='twilight_shifted',interpolation=None)
ax_5.set_xticks([])
ax_5.set_yticks([])
minv, maxv = im_5.get_clim() 
ax_5.set_title(f'(YZ-{im_slice}/{dimZ})',weight='bold',fontsize=6);

if SupportApplied:
    # Reconstructed electron densities with support applied
    yz_obj = recon_real_array[rec_num][-1][center-zoom:center+zoom,center-zoom:center+zoom,im_slice]
    xz_obj = recon_real_array[rec_num][-1][center-zoom:center+zoom,im_slice,center-zoom:center+zoom]
    xy_obj = recon_real_array[rec_num][-1][im_slice,center-zoom:center+zoom,center-zoom:center+zoom]

    fig_handle = plt.figure(constrained_layout = True, dpi = 250)
    fig_handle.patch.set_facecolor(f'white')
    spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
    plt.suptitle(f'Slices of object amplitude and phase: reconstruction {rec_num+1}')

    ax_0 = fig_handle.add_subplot(spec_handle[0,0])
    im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_0.set_xticks([])
    ax_0.set_yticks([])
    minv, maxv = im_0.get_clim()
    ax_0.set_title(f'(XY-{im_slice}/{dimX})',weight='bold',fontsize=6)
    c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05) 
    c_bar_0.set_label('electron density in a.u.', weight='bold',fontsize=6)
    c_bar_0.set_ticks([minv,maxv*0.5,maxv]) 

    ax_1 = fig_handle.add_subplot(spec_handle[0,1]) 
    im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_1.set_xticks([]) 
    ax_1.set_yticks([]) 
    minv, maxv = im_1.get_clim() 
    ax_1.set_title(f'(XZ-{im_slice}/{dimY})',weight='bold',fontsize=6)

    ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
    im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
    ax_2.set_xticks([]) 
    ax_2.set_yticks([]) 
    minv, maxv = im_2.get_clim()
    ax_2.set_title(f'(YZ-{im_slice}/{dimZ})',weight='bold',fontsize=6)

    # Reconstructed electron phases with support applied
    yz_diff = recon_phase_array[rec_num][-1][center-zoom:center+zoom,center-zoom:center+zoom,im_slice] * dens_scaling
    xz_diff = recon_phase_array[rec_num][-1][center-zoom:center+zoom,im_slice,center-zoom:center+zoom] * dens_scaling
    xy_diff = recon_phase_array[rec_num][-1][im_slice,center-zoom:center+zoom,center-zoom:center+zoom] * dens_scaling

    ax_3 = fig_handle.add_subplot(spec_handle[1,0]) 
    im_3 = plt.imshow(xy_diff,vmin=-pi,vmax=pi,cmap='twilight_shifted',interpolation=None)
    ax_3.set_xticks([])
    ax_3.set_yticks([]) 
    minv, maxv = im_3.get_clim()
    ax_3.set_title(f'(XY-{im_slice}/{dimX})',weight='bold',fontsize=6) 
    c_bar_3 = plt.colorbar(im_3, ax=ax_3,fraction=0.05) 
    c_bar_3.set_label('phase in radians', weight='bold',fontsize=6)
    c_bar_3.set_ticks([-3,0,3])

    ax_4 = fig_handle.add_subplot(spec_handle[1,1]) 
    im_4 = plt.imshow(xz_diff,vmin=-pi,vmax=pi,cmap='twilight_shifted',interpolation=None) 
    ax_4.set_xticks([]) 
    ax_4.set_yticks([]) 
    minv, maxv = im_4.get_clim() 
    ax_4.set_title(f'(XZ-{im_slice}/{dimY})',weight='bold',fontsize=6)

    ax_5 = fig_handle.add_subplot(spec_handle[1,2]) 
    im_5 = plt.imshow(yz_diff,vmin=-pi,vmax=pi,cmap='twilight_shifted',interpolation=None) 
    ax_5.set_xticks([])
    ax_5.set_yticks([])
    minv, maxv = im_5.get_clim() 
    ax_5.set_title(f'(YZ-{im_slice}/{dimZ})',weight='bold',fontsize=6);

# Reconstructed supports
yz_obj = support_array[rec_num][-1][center-zoom:center+zoom,center-zoom:center+zoom,im_slice]
xz_obj = support_array[rec_num][-1][center-zoom:center+zoom,im_slice,center-zoom:center+zoom]
xy_obj = support_array[rec_num][-1][im_slice,center-zoom:center+zoom,center-zoom:center+zoom]

fig_handle = plt.figure(constrained_layout = True, dpi = 250)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
plt.suptitle(f'Slices of object support: reconstruction {rec_num+1}')
cm = 'gray'
max_v = 1.0

ax_0 = fig_handle.add_subplot(spec_handle[0,0])
im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_0.set_xticks([])
ax_0.set_yticks([]) 
minv, maxv = im_0.get_clim()
ax_0.set_title(f'(XY-{im_slice}/{dimX})',weight='bold',fontsize=6) 
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.02, shrink=0.47)
c_bar_0.set_ticks([minv, maxv]) 

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_1.set_xticks([]) 
ax_1.set_yticks([]) 
minv, maxv = im_1.get_clim() 
ax_1.set_title(f'(XZ-{im_slice}/{dimY})',weight='bold',fontsize=6)

ax_2 = fig_handle.add_subplot(spec_handle[0,2])
im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_2.set_xticks([]) 
ax_2.set_yticks([]) 
minv, maxv = im_2.get_clim() 
ax_2.set_title(f'(YZ-{im_slice}/{dimZ})',weight='bold',fontsize=6);

im_slice = dimX//2
recon_fourier_array_masked = recon_fourier_array[rec_num][-1]
fourier_scaling = 1.0

yz_obj = recon_fourier_array_masked[:,:,im_slice] ** fourier_scaling
xz_obj = recon_fourier_array_masked[:,im_slice,:] ** fourier_scaling
xy_obj = recon_fourier_array_masked[im_slice,:,:] ** fourier_scaling

fig_handle = plt.figure(constrained_layout = True, dpi = 250)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 2, ncols = 3)
plt.suptitle(f'Slices of recovered Fourier intensity: reconstruction {rec_num+1}')
cm = 'viridis'
max_v = 0.3
write_text(f'{support_array[:,-1].sum(axis=(1,2,3))}')

# Reconstructed Fourier intensities
ax_0 = fig_handle.add_subplot(spec_handle[0,0])
im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_0.set_xticks([])
ax_0.set_yticks([]) 
minv, maxv = im_0.get_clim()
ax_0.set_title(f'(XY-{im_slice}/{dimX})',weight='bold',fontsize=6) 
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05) 
c_bar_0.set_ticks([minv,maxv*0.5,maxv]) 

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
ax_1.set_xticks([]) 
ax_1.set_yticks([])
minv, maxv = im_1.get_clim()
ax_1.set_title(f'(XZ-{im_slice}/{dimY})',weight='bold',fontsize=6) 

ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
ax_2.set_xticks([]) 
ax_2.set_yticks([])
minv, maxv = im_2.get_clim()
ax_2.set_title(f'(YZ-{im_slice}/{dimZ})',weight='bold',fontsize=6);

In [ ]:
iter_vec = np.linspace(0, niter_alg+niter_er, num = niter_store_errors)
plt.figure(7, dpi=150)
plt.plot(iter_vec, error_real_array.T,'b')
plt.plot(iter_vec, error_fourier_array.T,'r')
plt.yscale('log')
plt.xlabel('iteration #', weight='bold')
plt.ylabel('error', weight='bold');